# MVP v0.3.2.5b -- Alpha x Lambda LHS Sweep

**Builds on:** v0.3.2.5 (negative guidance sweep, best at lam=0) + v0.2.5.14d (alpha=0.4, rho=0.886)

**Goal:** Joint sweep of positive guidance strength (alpha) and negative guidance strength (lam)
using a 15-point Maximin Latin Hypercube design over alpha in [0.1, 0.5], lam/alpha in [0, 1].
This covers the 2D space efficiently in ~105 min instead of ~210 min for a 5x6 grid.

Formula: `guide = alpha * target_grad - lam * behavior_grad`

- `alpha` scales positive guidance (target policy)
- `lam` scales negative guidance (behavior policy), where `lam = ratio * alpha`
- `ratio = lam/alpha` in [0, 1] keeps negative proportional to positive

**Behavior policy:** BCGaussian (trained on same 200 demos + 50 rollouts as chunk diffuser)


In [ ]:
%matplotlib inline
import sys, os, importlib
import numpy as np
import torch
import torch.nn as nn
import h5py, json, math, time
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
CKPT_BASE = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models"
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
TARGET_ROLLOUT_DIR = PROJECT_ROOT / "rollouts" / "target_policy_50"
DIFFUSION_SAVE_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v0252_traj_mse"
BEHAVIOR_SAVE_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v0214_fixed_guidance"
ORACLE_JSON = PROJECT_ROOT / "results/2026-03-12/oracle_eval_all_checkpoints.json"
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

STATE_DIM = 19
ACTION_DIM = 7
TRANSITION_DIM = 26
CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84
CHUNK_SIZE = 4
N_DIFFUSION_STEPS = 256
BASE_DIM = 32
DIM_MULTS = (1, 4, 8)
ACTION_WEIGHT = 5.0
NUM_SYNTHETIC = 50
T_GEN = 60
GAMMA = 1.0
SCORE_TIMESTEP = 5
SIGMOID_SHARPNESS = 150.0

# Guidance config -- LHS sweep over (alpha, ratio) space
# 15-point Maximin Latin Hypercube: alpha in [0.1, 0.5], ratio (lam/alpha) in [0, 1]
# Generated with scipy.stats.qmc.LatinHypercube, seed optimized over 5000 draws
DESIGN_POINTS = [
    # (alpha, ratio) -> lam = alpha * ratio
    (0.14, 0.00),
    (0.11, 0.65),
    (0.15, 0.16),
    (0.18, 0.30),
    (0.22, 0.90),
    (0.24, 0.41),
    (0.26, 0.53),
    (0.29, 0.68),
    (0.33, 0.23),
    (0.35, 0.79),
    (0.38, 0.11),
    (0.42, 0.35),
    (0.42, 0.97),
    (0.47, 0.82),
    (0.49, 0.50),
]

TARGET_POLICIES = [
    {"name": "10demos_epoch10", "dir": "lift_diffusion_10demos/20260311115828", "ckpt": "models/model_epoch_10.pth"},
    {"name": "100demos_epoch20", "dir": "lift_diffusion_100demos/20260311135551", "ckpt": "models/model_epoch_20.pth"},
    {"name": "test_checkpoint", "dir": "test/20260309132349", "ckpt": "last.pth"},
    {"name": "10demos_epoch30", "dir": "lift_diffusion_10demos/20260311115828", "ckpt": "models/model_epoch_30.pth"},
    {"name": "50demos_epoch30", "dir": "lift_diffusion_50demos/20260311134204", "ckpt": "models/model_epoch_30.pth"},
    {"name": "200demos_epoch40", "dir": "lift_diffusion_200demos/20260311141036", "ckpt": "models/model_epoch_40.pth"},
]

print(f"LHS design: {len(DESIGN_POINTS)} points in alpha=[0.1,0.5] x ratio=[0,1]")
print(f"{len(TARGET_POLICIES)} policies x {len(DESIGN_POINTS)} design points = {len(TARGET_POLICIES)*len(DESIGN_POINTS)} guided runs")

In [ ]:
# ── Reward + OPE functions ──
def hard_reward(cube_z):
    return (cube_z > LIFT_THRESHOLD).astype(np.float32)

def sigmoid_reward(cube_z, center=LIFT_THRESHOLD, sharpness=SIGMOID_SHARPNESS):
    return (1.0 / (1.0 + np.exp(-sharpness * (cube_z - center)))).astype(np.float32)

def compute_ope_hard(states, gamma=1.0):
    cube_z = states[:, :, CUBE_Z_INDEX]
    return (hard_reward(cube_z) * (gamma ** np.arange(states.shape[1]))[None, :]).sum(axis=1)

def compute_ope_sigmoid(states, gamma=1.0):
    cube_z = states[:, :, CUBE_Z_INDEX]
    return (sigmoid_reward(cube_z) * (gamma ** np.arange(states.shape[1]))[None, :]).sum(axis=1)

def compute_sr_hard(states):
    cube_z = states[:, :, CUBE_Z_INDEX]
    return np.mean([np.any(cube_z[j] > LIFT_THRESHOLD) for j in range(states.shape[0])])

print("Functions defined.")

In [ ]:
# ── Load oracle ──
with open(ORACLE_JSON, "r") as f:
    oracle_all = json.load(f)

oracle_sr_map = {}
for pol in TARGET_POLICIES:
    name = pol["name"]
    if name == "test_checkpoint":
        with open(CKPT_BASE / "test/20260309132349/oracle_50.json", "r") as f:
            test_oracle = json.load(f)
        oracle_sr_map[name] = float(test_oracle["mean_return"])
    else:
        oracle_sr_map[name] = float(oracle_all[name]["mean_return"])

print("Oracle SR:")
for name, sr in sorted(oracle_sr_map.items(), key=lambda x: x[1]):
    print(f"  {name:<22} {sr*100:>5.0f}%")

In [ ]:
# ── Load data + normalization + diffuser ──
all_states_list, all_actions_list = [], []
target_data = []
for path in sorted(TARGET_ROLLOUT_DIR.glob("rollout_*.h5"))[:50]:
    with h5py.File(path, "r") as f:
        latents = f["latents"][:]
        actions = f["actions"][:]
    states = (latents[:, -1, :] if latents.ndim == 3 else latents).astype(np.float32)
    actions = actions.astype(np.float32)
    target_data.append({"states": states, "actions": actions})
    all_states_list.append(states)
    all_actions_list.append(actions)

with h5py.File(DEMO_HDF5, "r") as f:
    for dk in sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1])):
        demo = f[f"data/{dk}"]
        states = np.concatenate([demo["obs"][k][:].astype(np.float32) for k in OBS_KEYS], axis=-1)
        actions = demo["actions"][:].astype(np.float32)
        all_states_list.append(states)
        all_actions_list.append(actions)

all_states = np.concatenate(all_states_list, axis=0)
all_actions = np.concatenate(all_actions_list, axis=0)
norm_mean = np.concatenate([all_states.mean(0), all_actions.mean(0)]).astype(np.float32)
norm_std = np.maximum(np.concatenate([all_states.std(0), all_actions.std(0)]), 1e-6).astype(np.float32)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t

initial_states_t = torch.tensor(
    np.array([ep["states"][0] for ep in target_data[:NUM_SYNTHETIC]]),
    dtype=torch.float32, device=device
)

temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE, transition_dim=TRANSITION_DIM,
    dim=BASE_DIM, dim_mults=DIM_MULTS, attention=False,
).to(device)
diffusion_model = GaussianDiffusion(
    model=temporal_model, horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM, action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn, unnormalizer=unnormalize_fn,
    predict_epsilon=False, loss_type="l2",
    clip_denoised=False, action_weight=ACTION_WEIGHT,
).to(device)
ema = EMA(diffusion_model)
ema.ema_model.load_state_dict(
    torch.load(DIFFUSION_SAVE_DIR / "diffusion_model_ema.pt", map_location=device)
)
print(f"Loaded diffuser. Initial states: {initial_states_t.shape}")

In [ ]:
# ── Load BCGaussian behavior policy ──
class BCGaussian(nn.Module):
    """Simple BC policy with Gaussian output for SOPE-style guidance."""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)

    def forward(self, state):
        h = self.net(state)
        return self.mean_head(h), self.log_std_head(h).clamp(-5, 2)

    def log_prob(self, state, action):
        mean, log_std = self.forward(state)
        std = torch.exp(log_std)
        return -0.5 * (((action - mean) / std) ** 2 + 2 * log_std + math.log(2 * math.pi)).sum(dim=-1)

    def grad_log_prob(self, state, action):
        """Analytic gradient: -(a - mu) / sigma^2."""
        with torch.no_grad():
            mean, log_std = self.forward(state)
            std = torch.exp(log_std)
            return -(action - mean) / (std ** 2)

    def grad_log_prob_chunk(self, states, actions):
        B, T, _ = states.shape
        return self.grad_log_prob(
            states.reshape(B * T, -1), actions.reshape(B * T, -1)
        ).reshape(B, T, -1)

behavior_policy = BCGaussian(STATE_DIM, ACTION_DIM, hidden_dim=256).to(device)
behavior_policy.load_state_dict(
    torch.load(BEHAVIOR_SAVE_DIR / "bc_behavior.pt", map_location=device)
)
behavior_policy.eval()

# Sanity check
test_s = initial_states_t[:4]
test_a = torch.randn(4, ACTION_DIM, device=device)
test_grad = behavior_policy.grad_log_prob(test_s, test_a)
print(f"Behavior policy loaded. Test grad shape: {test_grad.shape}, magnitude: {test_grad.abs().mean():.4f}")

In [ ]:
# ── Build paired real trajectory array for state-based logRMSE ──
real_states_list = []
for ep in target_data[:NUM_SYNTHETIC]:
    s = ep["states"]
    T = len(s)
    if T >= T_GEN:
        real_states_list.append(s[:T_GEN])
    else:
        padded = np.zeros((T_GEN, STATE_DIM), dtype=np.float32)
        padded[:T] = s
        padded[T:] = s[-1]
        real_states_list.append(padded)
real_states = np.array(real_states_list)
print(f"Real states for comparison: {real_states.shape}")

In [ ]:
# ── Trajectory generator (positive + negative guidance) ──
def generate_trajectories(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim, chunk_size, t_gen, device,
    target_scorer=None, behavior_scorer=None,
    alpha=0.0, lam=0.0, normalize_grad=True,
):
    """
    SOPE-style guided trajectory generation.
    guide = alpha * target_grad - lam * behavior_grad
    Applied to action dimensions only.
    """
    use_pos = (target_scorer is not None and alpha > 0)
    use_neg = (behavior_scorer is not None and lam > 0)
    guided = use_pos or use_neg
    B = initial_states.shape[0]
    td = state_dim + action_dim
    n_ts = diffusion_model.n_timesteps

    pad = torch.cat([initial_states, torch.zeros(B, action_dim, device=device)], 1)
    cond_init = normalize_fn(pad)[:, :state_dim]
    all_traj = torch.zeros(B, t_gen, td, device=device)
    conditions = {0: cond_init}
    total = 0

    while total < t_gen:
        x = torch.randn(B, chunk_size, td, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_d in reversed(range(n_ts)):
            t_t = torch.full((B,), t_d, device=device, dtype=torch.long)
            with torch.no_grad():
                mm, _, mlv = diffusion_model.p_mean_variance(x=x, t=t_t)
                ms = torch.exp(0.5 * mlv)

            if guided:
                mm = unnormalize_fn(mm)
                sc = mm[:, :, :state_dim]
                ac = mm[:, :, state_dim:]

                # Positive guidance: grad_log_prob of target policy
                pos_grad = torch.zeros(B, chunk_size, action_dim, device=device)
                if use_pos:
                    tg = target_scorer.grad_log_prob_chunk(sc, ac)
                    if normalize_grad:
                        tg = tg / (tg.norm(dim=-1, keepdim=True) + 1e-6)
                    pos_grad = tg

                # Negative guidance: grad_log_prob of behavior policy
                neg_grad = torch.zeros(B, chunk_size, action_dim, device=device)
                if use_neg:
                    bg = behavior_scorer.grad_log_prob_chunk(sc, ac)
                    if normalize_grad:
                        bg = bg / (bg.norm(dim=-1, keepdim=True) + 1e-6)
                    neg_grad = bg

                # Combined: alpha * target - lam * behavior (independent scales)
                guide = torch.zeros_like(mm)
                guide[:, :, state_dim:] = alpha * pos_grad - lam * neg_grad
                mm = mm + guide

                mm = normalize_fn(mm)
                mm = apply_conditioning(mm, conditions, state_dim)
                mm = unnormalize_fn(mm)
                mm = normalize_fn(mm)

            noise = torch.randn_like(x)
            x = mm + (1 - (t_d == 0) * 1.0) * ms * noise
            x = apply_conditioning(x, conditions, state_dim)

        chunk_u = unnormalize_fn(x)
        n_store = min(chunk_size - 1, t_gen - total)
        all_traj[:, total:total+n_store] = chunk_u[:, 1:1+n_store]
        total += n_store

        if total < t_gen:
            last_state_norm = x[:, -1, :state_dim]
            conditions = {0: last_state_norm}

    return all_traj.cpu().numpy()

print("Generator ready (positive + negative guidance).")

In [ ]:
# ── Pre-load all target scorers (load each checkpoint once) ──
target_scorers = {}
for pol in TARGET_POLICIES:
    name = pol["name"]
    run_dir = CKPT_BASE / pol["dir"]
    print(f"Loading {name}...", end=" ", flush=True)
    t0 = time.time()
    ckpt = load_checkpoint(run_dir, ckpt_path=Path(pol["ckpt"]))
    target_algo = build_algo_from_checkpoint(ckpt, device=str(device))
    target_scorers[name] = RobomimicDiffusionScorer(
        target_algo, device=str(device), score_timestep=SCORE_TIMESTEP, obs_keys=OBS_KEYS
    )
    print(f"{time.time()-t0:.0f}s")
print(f"\nAll {len(target_scorers)} scorers loaded.")


In [ ]:
# ── Generate unguided baseline (once) ──
print("Generating unguided trajectories...")
np.random.seed(42)
torch.manual_seed(42)
t0 = time.time()
unguided_trajs = generate_trajectories(
    diffusion_model=ema.ema_model, initial_states=initial_states_t,
    normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
    state_dim=STATE_DIM, action_dim=ACTION_DIM,
    chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
)
unguided_states = unguided_trajs[:, :, :STATE_DIM]
unguided_sr = compute_sr_hard(unguided_states)
unguided_hard = compute_ope_hard(unguided_states, GAMMA)
unguided_sigm = compute_ope_sigmoid(unguided_states, GAMMA)
unguided_per_traj_mse = np.mean((real_states - unguided_states) ** 2, axis=(1, 2))
unguided_state_rmse = np.sqrt(np.mean(unguided_per_traj_mse))
unguided_state_log_rmse = np.log(unguided_state_rmse + 1e-8)
print(f"Unguided: SR={unguided_sr*100:.0f}%, hard={unguided_hard.mean():.3f}, "
      f"sigmoid={unguided_sigm.mean():.3f}, state_logRMSE={unguided_state_log_rmse:.4f}, {time.time()-t0:.0f}s")

In [ ]:
# ── Sweep: (alpha, ratio) design points x target_policy ──
sweep_results = {}  # key: (design_idx, policy_name)
t0_all = time.time()
total_runs = len(DESIGN_POINTS) * len(TARGET_POLICIES)
run_count = 0

for di, (alpha, ratio) in enumerate(DESIGN_POINTS):
    lam = alpha * ratio
    print(f"\n{'='*80}")
    print(f"DESIGN POINT {di+1}/{len(DESIGN_POINTS)}: alpha={alpha:.2f}, ratio={ratio:.2f}, lam={lam:.3f}")
    print(f"{'='*80}")

    for pol in TARGET_POLICIES:
        name = pol["name"]
        run_count += 1
        oracle_sr = oracle_sr_map[name]

        print(f"  [{run_count}/{total_runs}] {name} (oracle={oracle_sr*100:.0f}%)...", end=" ", flush=True)
        np.random.seed(42)
        torch.manual_seed(42)
        t0 = time.time()

        guided_trajs = generate_trajectories(
            diffusion_model=ema.ema_model, initial_states=initial_states_t,
            normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
            state_dim=STATE_DIM, action_dim=ACTION_DIM,
            chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
            target_scorer=target_scorers[name],
            behavior_scorer=behavior_policy,
            alpha=alpha, lam=lam,
            normalize_grad=True,
        )
        gen_time = time.time() - t0

        guided_states = guided_trajs[:, :, :STATE_DIM]
        sr = compute_sr_hard(guided_states)
        ope_hard = float(compute_ope_hard(guided_states, GAMMA).mean())
        ope_sigm = float(compute_ope_sigmoid(guided_states, GAMMA).mean())

        per_traj_mse = np.mean((real_states - guided_states) ** 2, axis=(1, 2))
        state_rmse = np.sqrt(np.mean(per_traj_mse))
        state_log_rmse = np.log(state_rmse + 1e-8)

        sweep_results[(di, name)] = {
            "alpha": alpha,
            "ratio": ratio,
            "lam": lam,
            "oracle_sr": oracle_sr,
            "guided_sr": sr,
            "ope_hard": ope_hard,
            "ope_sigmoid": ope_sigm,
            "state_rmse": state_rmse,
            "state_log_rmse": state_log_rmse,
        }
        print(f"{gen_time:.0f}s -- SR={sr*100:.0f}%, hard={ope_hard:.3f}, sigm={ope_sigm:.3f}, logRMSE={state_log_rmse:.4f}")

total_time = time.time() - t0_all
print(f"\nTotal sweep: {total_time:.0f}s ({total_time/60:.1f} min)")


In [ ]:
# ── Compute metrics per design point ──
policy_names = [p["name"] for p in TARGET_POLICIES]
oracle_srs = np.array([oracle_sr_map[n] for n in policy_names])

design_metrics = {}
for di, (alpha, ratio) in enumerate(DESIGN_POINTS):
    lam = alpha * ratio
    opes_hard = np.array([sweep_results[(di, n)]["ope_hard"] for n in policy_names])
    opes_sigm = np.array([sweep_results[(di, n)]["ope_sigmoid"] for n in policy_names])
    guided_srs = np.array([sweep_results[(di, n)]["guided_sr"] for n in policy_names])

    opes_hard_norm = opes_hard / T_GEN
    opes_sigm_norm = opes_sigm / T_GEN

    rho_h, p_h = stats.spearmanr(oracle_srs, opes_hard)
    rho_s, p_s = stats.spearmanr(oracle_srs, opes_sigm)

    eps = 1e-6
    rmse_h = np.sqrt(np.mean((oracle_srs - opes_hard_norm) ** 2))
    rmse_s = np.sqrt(np.mean((oracle_srs - opes_sigm_norm) ** 2))
    log_rmse_h = np.sqrt(np.mean((np.log(oracle_srs + eps) - np.log(opes_hard_norm + eps)) ** 2))
    log_rmse_s = np.sqrt(np.mean((np.log(oracle_srs + eps) - np.log(opes_sigm_norm + eps)) ** 2))

    state_log_rmses = np.array([sweep_results[(di, n)]["state_log_rmse"] for n in policy_names])
    mean_state_log_rmse = float(np.mean(state_log_rmses))

    def regret_at_k(oracle_vals, ope_vals, k):
        true_topk = np.argsort(oracle_vals)[-k:]
        est_topk = np.argsort(ope_vals)[-k:]
        return float(oracle_vals[true_topk].mean() - oracle_vals[est_topk].mean())

    design_metrics[di] = {
        "alpha": alpha, "ratio": ratio, "lam": lam,
        "rho_hard": rho_h, "rho_sigm": rho_s,
        "rmse_hard": rmse_h, "rmse_sigm": rmse_s,
        "log_rmse_hard": log_rmse_h, "log_rmse_sigm": log_rmse_s,
        "regret1_hard": regret_at_k(oracle_srs, opes_hard, 1),
        "regret3_hard": regret_at_k(oracle_srs, opes_hard, 3),
        "regret1_sigm": regret_at_k(oracle_srs, opes_sigm, 1),
        "regret3_sigm": regret_at_k(oracle_srs, opes_sigm, 3),
        "mean_state_log_rmse": mean_state_log_rmse,
    }

print("Metrics computed for all design points.")


In [ ]:
# ── Summary table ──
print("=" * 130)
print("v0.3.2.5b: ALPHA x LAMBDA LHS SWEEP (15 Maximin LHS points)")
print("=" * 130)

# Per-design-point per-policy table
for di, (alpha, ratio) in enumerate(DESIGN_POINTS):
    m = design_metrics[di]
    lam = alpha * ratio
    print(f"\n--- #{di+1}: alpha={alpha:.2f}, ratio={ratio:.2f}, lam={lam:.3f} | rho_hard={m['rho_hard']:+.3f} rho_sigm={m['rho_sigm']:+.3f} ---")
    for name in policy_names:
        r = sweep_results[(di, name)]
        print(f"  {name:<22} oracle={r['oracle_sr']*100:>5.0f}%  SR={r['guided_sr']*100:>5.0f}%  "
              f"hard={r['ope_hard']:>6.3f}  sigm={r['ope_sigmoid']:>6.3f}  logRMSE={r['state_log_rmse']:>7.4f}")

# Aggregate comparison
print("\n" + "=" * 130)
print("AGGREGATE METRICS BY DESIGN POINT")
print("=" * 130)
col_hdr = ("{:<4} {:>5} {:>5} {:>6} {:>7} {:>7} {:>8} {:>8} "
           "{:>10} {:>10} {:>10} {:>7} {:>7} {:>7} {:>7}").format(
    "#", "alpha", "ratio", "lam", "rho_H", "rho_S", "RMSE_H", "RMSE_S",
    "logRMSE_H", "logRMSE_S", "stateLogR", "R@1_H", "R@3_H", "R@1_S", "R@3_S")
print("\n" + col_hdr)
print("-" * 130)
for di in range(len(DESIGN_POINTS)):
    m = design_metrics[di]
    print(f"{di+1:<4} {m['alpha']:>5.2f} {m['ratio']:>5.2f} {m['lam']:>6.3f} "
          f"{m['rho_hard']:>+7.3f} {m['rho_sigm']:>+7.3f} "
          f"{m['rmse_hard']:>8.4f} {m['rmse_sigm']:>8.4f} "
          f"{m['log_rmse_hard']:>10.4f} {m['log_rmse_sigm']:>10.4f} "
          f"{m['mean_state_log_rmse']:>10.4f} "
          f"{m['regret1_hard']:>7.3f} {m['regret3_hard']:>7.3f} "
          f"{m['regret1_sigm']:>7.3f} {m['regret3_sigm']:>7.3f}")

best_di = max(range(len(DESIGN_POINTS)), key=lambda i: design_metrics[i]["rho_hard"])
best_m = design_metrics[best_di]
print(f"\nBest (by rho_hard): #{best_di+1} alpha={best_m['alpha']:.2f}, ratio={best_m['ratio']:.2f}, "
      f"lam={best_m['lam']:.3f} -> rho={best_m['rho_hard']:+.3f}")


In [ ]:
# ── Figure 1: Metrics over (alpha, ratio) design space ──
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

alphas = np.array([design_metrics[i]["alpha"] for i in range(len(DESIGN_POINTS))])
ratios = np.array([design_metrics[i]["ratio"] for i in range(len(DESIGN_POINTS))])

metrics_to_plot = [
    ("rho_hard", "Spearman rho (hard)", "RdYlGn", True),
    ("rho_sigm", "Spearman rho (sigmoid)", "RdYlGn", True),
    ("rmse_hard", "RMSE (hard)", "RdYlGn_r", False),
    ("mean_state_log_rmse", "State log(RMSE)", "RdYlGn_r", False),
]

for ax, (key, title, cmap, higher_better) in zip(axes.flat, metrics_to_plot):
    vals = np.array([design_metrics[i][key] for i in range(len(DESIGN_POINTS))])
    sc = ax.scatter(alphas, ratios, c=vals, cmap=cmap, s=200, edgecolor="black", zorder=5)
    for i in range(len(DESIGN_POINTS)):
        ax.annotate(f"{vals[i]:.3f}", (alphas[i], ratios[i]),
                    textcoords="offset points", xytext=(6, 6), fontsize=7)
    plt.colorbar(sc, ax=ax)
    # Mark best point
    best_i = np.argmax(vals) if higher_better else np.argmin(vals)
    ax.scatter([alphas[best_i]], [ratios[best_i]], s=300, facecolors="none",
              edgecolors="red", linewidths=2, zorder=6)
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Ratio (lam / alpha)")
    ax.set_title(title)
    ax.set_xlim(0.05, 0.55)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.2)

fig.suptitle("v0.3.2.5b: Metrics over (alpha, ratio) Design Space", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 2: OPE scatter for best/worst design points ──
sorted_by_rho = sorted(range(len(DESIGN_POINTS)), key=lambda i: design_metrics[i]["rho_hard"], reverse=True)
show_indices = sorted_by_rho[:3] + sorted_by_rho[-3:]  # top 3 + bottom 3

fig, axes = plt.subplots(1, len(show_indices), figsize=(4*len(show_indices), 4), sharey=True)

for ax, di in zip(axes, show_indices):
    m = design_metrics[di]
    opes = np.array([sweep_results[(di, n)]["ope_hard"] / T_GEN for n in policy_names])
    ax.scatter(oracle_srs * 100, opes * 100, s=80, c="coral", edgecolor="black", zorder=5)
    ax.plot([0, 100], [0, 100], "k--", alpha=0.3)
    for j, n in enumerate(policy_names):
        label = n.replace("demos_epoch", "e").replace("test_checkpoint", "test")
        ax.annotate(label, (oracle_srs[j]*100, opes[j]*100),
                    textcoords="offset points", xytext=(3, 3), fontsize=5)
    ax.set_xlabel("Oracle (%)")
    ax.set_title(f"#{di+1}: a={m['alpha']:.2f} r={m['ratio']:.2f}\nrho={m['rho_hard']:+.3f}", fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5, 100)
    ax.set_ylim(-5, 100)

axes[0].set_ylabel("OPE Hard (normalized %)")
fig.suptitle("v0.3.2.5b: OPE vs Oracle — Best 3 + Worst 3 Design Points", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 3: State logRMSE across design space ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: logRMSE vs rho_hard (tradeoff)
ax = axes[0]
rhos = np.array([design_metrics[i]["rho_hard"] for i in range(len(DESIGN_POINTS))])
lrs = np.array([design_metrics[i]["mean_state_log_rmse"] for i in range(len(DESIGN_POINTS))])
ax.scatter(rhos, lrs, s=100, c=ratios, cmap="coolwarm", edgecolor="black", zorder=5)
for i in range(len(DESIGN_POINTS)):
    ax.annotate(f"a={alphas[i]:.2f}\nr={ratios[i]:.2f}", (rhos[i], lrs[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=6)
ax.axhline(unguided_state_log_rmse, color="gray", ls="--", alpha=0.7, label=f"Unguided ({unguided_state_log_rmse:.3f})")
sc = ax.scatter(rhos, lrs, s=100, c=ratios, cmap="coolwarm", edgecolor="black", zorder=5)
plt.colorbar(sc, ax=ax, label="Ratio")
ax.set_xlabel("Spearman rho (hard)")
ax.set_ylabel("State log(RMSE)")
ax.set_title("Rank Correlation vs Trajectory Realism")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 2: logRMSE colored by alpha
ax = axes[1]
sc = ax.scatter(alphas, lrs, s=100, c=ratios, cmap="coolwarm", edgecolor="black", zorder=5)
for i in range(len(DESIGN_POINTS)):
    ax.annotate(f"r={ratios[i]:.2f}", (alphas[i], lrs[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.axhline(unguided_state_log_rmse, color="gray", ls="--", alpha=0.7, label="Unguided")
plt.colorbar(sc, ax=ax, label="Ratio")
ax.set_xlabel("Alpha")
ax.set_ylabel("State log(RMSE)")
ax.set_title("State logRMSE vs Alpha (colored by ratio)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle("v0.3.2.5b: State-based logRMSE", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
